In [ ]:
import evidently

from evidently import Report
from evidently.presets import DataDriftPreset
import pandas as pd

from src.constants import DATA_DIR

In [ ]:
reference_df = pd.read_parquet(DATA_DIR / "processed" / "val_dataset.parquet")
current_df = pd.read_parquet(DATA_DIR / "processed" / "drifted_val_dataset.parquet")

target_cols_ref = [col for col in reference_df.columns if "target" in col.lower()]
target_cols_curr = [col for col in current_df.columns if "target" in col.lower()]

if target_cols_ref:
    reference_df = reference_df.drop(columns=target_cols_ref)
if target_cols_curr:
    current_df = current_df.drop(columns=target_cols_curr)

threshold = 0.2
report = Report(metrics=[DataDriftPreset(drift_share=threshold)])

In [ ]:
report_run = report.run(reference_data=reference_df, current_data=current_df)

In [36]:

report_dict = report_run.dict()


In [37]:
drift_detected = report_dict["metrics"][0]["value"]["share"] > threshold

# Extract feature-level drift scores from ValueDrift metrics
feature_drifts = {}
for metric in report_dict["metrics"]:
    if "ValueDrift(column=" in metric["metric_id"]:
        # Extract column name from metric_id (e.g., "ValueDrift(column=f0)" -> "f0")
        column_name = metric["metric_id"].split("column=")[1].rstrip(")")
        drift_score = float(metric["value"])
        feature_drifts[column_name] = drift_score

# Select up to 3 features with highest drift scores
sorted_features = sorted(feature_drifts.items(), key=lambda x: x[1], reverse=True)
all_features = dict(sorted_features)
selected_features = dict(sorted_features[:3])

overall_drift_score = sum(all_features.values()) / len(all_features) if all_features else 0

output = {
    "drift_detected": drift_detected,
    "feature_drifts": selected_features,
    "overall_drift_score": overall_drift_score,
}

In [38]:
output

{'drift_detected': True,
 'feature_drifts': {'f1684': 1.51298636479256,
  'f1243': 1.5121849873118525,
  'f1705': 1.5112067735620889},
 'overall_drift_score': 0.1409907783024059}

In [29]:
output

{'drift_detected': True,
 'feature_drifts': {'f1684': 1.51298636479256,
  'f1243': 1.5121849873118525,
  'f1705': 1.5112067735620889},
 'overall_drift_score': 1.5121260418888338}